<a href="https://colab.research.google.com/github/jsl5710/audio_hallucination/blob/claude%2Fcreate-branch-structure-YBUPZ/Hallucination_audio_text_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/ICASSP_Data/
%ls /content/drive/MyDrive/ICASSP_Data/

[Errno 2] No such file or directory: '/content/drive/MyDrive/ICASSP_Data/'
/content/drive/MyDrive/ICASSP_Hallucinaton
ls: cannot access '/content/drive/MyDrive/ICASSP_Data/': No such file or directory


In [ ]:
%pip uninstall transformers
%pip install git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview
%pip accelerate soundfile pandas tqdm
# %pip install flash-attn --no-build-isolation  # Optional for speed

Found existing installation: transformers 4.52.0.dev0
Uninstalling transformers-4.52.0.dev0:
  Would remove:
    /usr/local/bin/transformers-cli
    /usr/local/lib/python3.12/dist-packages/transformers-4.52.0.dev0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/transformers/*
Proceed (Y/n)? y
  Successfully uninstalled transformers-4.52.0.dev0
  Cloning https://github.com/huggingface/transformers (to revision v4.51.3-Qwen2.5-Omni-preview) to /tmp/pip-req-build-huzgpqz5
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-huzgpqz5
  Running command git checkout -q f551466201fb4089d6df9dc55d80f0edbd149d85
  Resolved https://github.com/huggingface/transformers to commit f551466201fb4089d6df9dc55d80f0edbd149d85
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.52.0.dev0-py3-non

ERROR: unknown command "accelerate"


In [ ]:
# It's highly recommended to use `[decord]` feature for faster video loading.
%pip install qwen-omni-utils[decord] -U

# Data

In [ ]:
# %ls /content/drive/MyDrive/ICASSP_Hallucinaton/Audio_data/
%cd /content/drive/MyDrive/ICASSP_Hallucinaton/

/content/drive/MyDrive/ICASSP_Hallucinaton


# Main

In [ ]:
import torch
import pandas as pd
import json
import os
import time
import hashlib
import pickle
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Set
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info
from tqdm import tqdm

# Set CUDA memory management environment variable
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ========================== CONFIGURATION ==========================

# Base data directory
BASE_DATA_DIR = "/content/drive/MyDrive/ICASSP_Hallucinaton/Audio_data"

# Language folders
LANGUAGE_FOLDERS = {
    'english': 'English',
    'kazakh': 'Kazakh',
    'russian': 'Russian'
}

# Models to test
MODELS = [
    "Qwen/Qwen2.5-Omni-3B",
    # "Qwen/Qwen2.5-Omni-7B",
]

# Experiment types
EXPERIMENT_TYPES = ['audio', 'text']

# Output directory structure
OUTPUT_DIR = "hallucination_results"
CHECKPOINT_DIR = "checkpoints"

# ========================== PROMPTS ==========================

# System prompt for audio processing
SYSTEM_PROMPT = "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech."

# Audio prompts
AUDIO_DIRECT_PROMPT = """You are an expert assistant specialized in analyzing audio content and detecting hallucinations. Your task is to identify whether there are any hallucinations in the audio, determine their type, and assess their severity level.

Hallucination Types:
- Factual Contradiction: Statements that directly conflict with known facts or information provided in the audio content.
- Factual Fabrication: Insertion of fabricated yet plausible-sounding details not grounded in the audio content.
- Contextual Inconsistency: Subtle alterations that distort the meaning, emphasis, or context of the audio content without introducing explicit factual errors.

Severity Levels:
- Mild: Subtle distortions or minor deviations that preserve the main narrative and plausibility of the audio content.
- Moderate: Noticeable inconsistencies or factual alterations that affect key details or context while maintaining partial alignment with the audio content.
- Severe: Major contradictions, fabrications, or contextual breakdowns that substantially misrepresent or conflict with the audio content's facts or intent.

Given the audio content, classify potential hallucinations:

1. BINARY: Is there hallucination? (yes/no)
2. TYPE: If yes, what type? (factual_contradiction/factual_fabrication/contextual_inconsistency/none)
3. DEGREE: If yes, what degree? (mild/moderate/severe/none)

Output format: {{"binary": "yes/no", "type": "factual_contradiction/factual_fabrication/contextual_inconsistency/none", "degree": "mild/moderate/severe/none"}}"""

AUDIO_COT_PROMPT = """You are an expert assistant specialized in analyzing audio content and detecting hallucinations. Your task is to identify whether there are any hallucinations in the audio, determine their type, and assess their severity level.

Hallucination Types:
- Factual Contradiction: Statements that directly conflict with known facts or information provided in the audio content.
- Factual Fabrication: Insertion of fabricated yet plausible-sounding details not grounded in the audio content.
- Contextual Inconsistency: Subtle alterations that distort the meaning, emphasis, or context of the audio content without introducing explicit factual errors.

Severity Levels:
- Mild: Subtle distortions or minor deviations that preserve the main narrative and plausibility of the audio content.
- Moderate: Noticeable inconsistencies or factual alterations that affect key details or context while maintaining partial alignment with the audio content.
- Severe: Major contradictions, fabrications, or contextual breakdowns that substantially misrepresent or conflict with the audio content's facts or intent.

Given the audio content, classify potential hallucinations, think step-by-step:

Then classify:
1. BINARY: Is there hallucination? (yes/no)
2. TYPE: If yes, what type? (factual_contradiction/factual_fabrication/contextual_inconsistency/none)
3. DEGREE: If yes, what degree? (mild/moderate/severe/none)

Output format: {{"binary": "yes/no", "type": "factual_contradiction/factual_fabrication/contextual_inconsistency/none", "degree": "mild/moderate/severe/none"}}"""

# Text prompts
TEXT_DIRECT_PROMPT = AUDIO_DIRECT_PROMPT.replace("audio content", "text content").replace("article content", "text content")
TEXT_COT_PROMPT = AUDIO_COT_PROMPT.replace("audio content", "text content").replace("article content", "text content")

# ========================== SETUP ==========================

def setup_model(model_name: str, use_flash_attn: bool = False):
    """Load and setup Qwen2.5-Omni model"""
    print(f"Loading model: {model_name}")

    kwargs = {
        "torch_dtype": "auto",
        "device_map": "auto"
    }

    if use_flash_attn:
        try:
            print("Attempting to use Flash Attention 2...")
            kwargs["attn_implementation"] = "flash_attention_2"
            model = Qwen2_5OmniForConditionalGeneration.from_pretrained(model_name, **kwargs)
            print("Flash Attention 2 loaded successfully")
        except Exception as e:
            print(f"Flash Attention 2 failed: {e}")
            print("Falling back to standard attention...")
            kwargs.pop("attn_implementation", None)
            model = Qwen2_5OmniForConditionalGeneration.from_pretrained(model_name, **kwargs)
            print("Standard attention loaded successfully")
    else:
        print("Using standard attention (Flash Attention 2 disabled)")
        model = Qwen2_5OmniForConditionalGeneration.from_pretrained(model_name, **kwargs)
        print("Model loaded successfully")

    processor = Qwen2_5OmniProcessor.from_pretrained(model_name)

    # Disable audio output to save memory (~2GB)
    model.disable_talker()

    return model, processor

# ========================== DATA LOADING ==========================

def load_transcription_data(language_folder: str, language: str, filter_hallucinated_only: bool = False) -> pd.DataFrame:
    """Load transcription file from language folder with optional hallucination filtering"""

    folder_path = Path(BASE_DATA_DIR) / language_folder

    if not folder_path.exists():
        raise FileNotFoundError(f"Language folder not found: {folder_path}")

    # Find CSV file in the folder
    csv_files = list(folder_path.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in {folder_path}")

    if len(csv_files) > 1:
        print(f"Multiple CSV files found in {folder_path}, using: {csv_files[0]}")

    csv_file = csv_files[0]

    try:
        df = pd.read_csv(csv_file, encoding='utf-8')
        print(f"Loaded {csv_file}")
    except Exception as e:
        try:
            df = pd.read_csv(csv_file, sep='\t', encoding='utf-8')
            print(f"Loaded {csv_file} (tab-separated)")
        except Exception as e2:
            raise Exception(f"Failed to load {csv_file}: {e}, {e2}")

    print(f"Original data shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    # Verify required columns exist
    required_columns = ['filename', 'text', 'hallucination', 'hallucination_type', 'hallucination_level']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns in {csv_file}: {missing_columns}")

    # Apply filtering if requested
    if filter_hallucinated_only:
        filtered_df = df[df['hallucination'].str.lower() == 'yes'].copy()
        print(f"Filtered data (hallucination=yes): {filtered_df.shape}")
        processed_df = filtered_df
    else:
        print(f"Processing all data: {df.shape}")
        processed_df = df.copy()

    # Add language and folder path columns
    processed_df['language'] = language
    processed_df['language_folder'] = language_folder

    # Reset index to ensure consistent indexing
    processed_df.reset_index(drop=True, inplace=True)

    return processed_df

def load_all_transcription_data(filter_hallucinated_only: bool = False) -> Dict[str, pd.DataFrame]:
    """Load all transcription files with optional hallucination filtering"""
    all_data = {}

    for language, folder_name in LANGUAGE_FOLDERS.items():
        filter_msg = "hallucinated samples only" if filter_hallucinated_only else "all samples"
        print(f"\nLoading {language} transcriptions from {folder_name}/ ({filter_msg})...")
        try:
            data = load_transcription_data(folder_name, language, filter_hallucinated_only)
            all_data[language] = data
        except Exception as e:
            print(f"Failed to load {language} data: {e}")

    return all_data

# ========================== SMART CHECKPOINT MANAGER ==========================

class SmartCheckpointManager:
    def __init__(self, model_name: str, language: str, experiment_type: str, output_dir: str = OUTPUT_DIR):
        self.model_name = model_name.split('/')[-1]
        self.language = language
        self.experiment_type = experiment_type
        self.output_dir = Path(output_dir) / self.model_name / experiment_type
        self.checkpoint_dir = Path(CHECKPOINT_DIR) / self.model_name / experiment_type

        # Create directories
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        # File paths
        self.results_file = self.output_dir / f"{language}_results.csv"
        self.checkpoint_file = self.checkpoint_dir / f"{language}_checkpoint.pkl"
        self.progress_file = self.checkpoint_dir / f"{language}_progress.json"
        self.sample_mapping_file = self.checkpoint_dir / f"{language}_sample_mapping.pkl"

    def _create_sample_key(self, row: pd.Series) -> str:
        """Create a unique key for a sample based on filename and text content"""
        filename = str(row.get('filename', ''))
        text = str(row.get('text', ''))[:100]  # First 100 chars to avoid huge keys
        language = str(row.get('language', ''))

        # Create a hash of the combination for uniqueness
        key_string = f"{filename}|{text}|{language}"
        return hashlib.md5(key_string.encode('utf-8')).hexdigest()

    def _create_sample_mapping(self, df: pd.DataFrame) -> Dict[str, int]:
        """Create mapping from sample keys to current dataframe indices"""
        mapping = {}
        for idx, row in df.iterrows():
            key = self._create_sample_key(row)
            mapping[key] = idx
        return mapping

    def _load_existing_sample_mapping(self) -> Dict[str, Dict]:
        """Load existing sample mapping with processed results"""
        if self.sample_mapping_file.exists():
            try:
                with open(self.sample_mapping_file, 'rb') as f:
                    return pickle.load(f)
            except Exception as e:
                print(f"Warning: Failed to load sample mapping: {e}")
        return {}

    def _save_sample_mapping(self, mapping: Dict[str, Dict]):
        """Save sample mapping with processed results"""
        try:
            with open(self.sample_mapping_file, 'wb') as f:
                pickle.dump(mapping, f)
        except Exception as e:
            print(f"Warning: Failed to save sample mapping: {e}")

    def smart_load_existing_results(self, current_df: pd.DataFrame) -> Tuple[pd.DataFrame, Set[int]]:
        """
        Intelligently load existing results and map them to the current dataset
        Returns: (results_dataframe, set_of_processed_indices)
        """
        # Initialize results dataframe
        results_df = self._initialize_results_df(current_df)
        processed_indices = set()

        # Create mapping for current dataset
        current_mapping = self._create_sample_mapping(current_df)

        print(f"Current dataset has {len(current_mapping)} unique samples")

        # Load existing results if they exist
        if not self.results_file.exists():
            print("No existing results found - starting fresh")
            return results_df, processed_indices

        try:
            existing_df = pd.read_csv(self.results_file, encoding='utf-8')
            print(f"Found existing results with {len(existing_df)} samples")

            matched_samples = 0
            unmatched_samples = 0

            # Try to match existing results to current dataset
            for existing_idx, existing_row in existing_df.iterrows():
                # Create key for existing sample
                existing_key = self._create_sample_key(existing_row)

                # Check if this sample exists in current dataset
                if existing_key in current_mapping:
                    current_idx = current_mapping[existing_key]

                    # Copy all prediction columns from existing results
                    pred_columns = [col for col in existing_df.columns if col.startswith('pred_')]

                    # Check if this sample was actually processed (has non-empty predictions)
                    is_processed = self._check_sample_processed(existing_row, pred_columns)

                    if is_processed:
                        for col in pred_columns:
                            if col in existing_df.columns:
                                results_df.loc[current_idx, col] = existing_row[col]

                        processed_indices.add(current_idx)
                        matched_samples += 1
                    else:
                        unmatched_samples += 1
                else:
                    unmatched_samples += 1

            print(f"Successfully matched {matched_samples} processed samples")
            print(f"Found {unmatched_samples} samples that couldn't be matched or weren't processed")

            # Save updated sample mapping for future use
            updated_mapping = {}
            for idx, row in results_df.iterrows():
                if idx in processed_indices:
                    key = self._create_sample_key(row)
                    pred_data = {}
                    for col in results_df.columns:
                        if col.startswith('pred_'):
                            pred_data[col] = results_df.loc[idx, col]
                    updated_mapping[key] = {
                        'index': idx,
                        'predictions': pred_data,
                        'processed': True
                    }

            self._save_sample_mapping(updated_mapping)

        except Exception as e:
            print(f"Error loading existing results: {e}")
            print("Starting fresh...")

        return results_df, processed_indices

    def _check_sample_processed(self, row: pd.Series, pred_columns: List[str]) -> bool:
        """Check if a sample has been fully processed based on prediction columns"""
        required_cols = ['pred_direct_binary', 'pred_cot_binary']

        for col in required_cols:
            if col not in pred_columns:
                return False
            value = row.get(col, '')
            if pd.isna(value) or str(value).strip() == '' or str(value).lower() == 'nan':
                return False

        return True

    def save_checkpoint(self, df: pd.DataFrame, processed_indices: set, current_idx: int):
        """Save checkpoint with current progress"""
        checkpoint_data = {
            'processed_indices': processed_indices,
            'current_idx': current_idx,
            'timestamp': datetime.now().isoformat(),
            'total_samples': len(df),
            'experiment_type': self.experiment_type,
            'smart_resume': True
        }

        # Save checkpoint metadata
        with open(self.checkpoint_file, 'wb') as f:
            pickle.dump(checkpoint_data, f)

        # Save progress info (human readable)
        progress_info = {
            'model': self.model_name,
            'language': self.language,
            'experiment_type': self.experiment_type,
            'processed_count': len(processed_indices),
            'total_count': len(df),
            'progress_percentage': (len(processed_indices) / len(df)) * 100 if len(df) > 0 else 0,
            'last_updated': datetime.now().isoformat(),
            'smart_resume_enabled': True
        }

        with open(self.progress_file, 'w') as f:
            json.dump(progress_info, f, indent=2)

        # Save current results
        df.to_csv(self.results_file, index=False, encoding='utf-8')

        # Update sample mapping
        sample_mapping = {}
        for idx in processed_indices:
            if idx < len(df):
                key = self._create_sample_key(df.iloc[idx])
                pred_data = {}
                for col in df.columns:
                    if col.startswith('pred_'):
                        pred_data[col] = df.loc[idx, col]
                sample_mapping[key] = {
                    'index': idx,
                    'predictions': pred_data,
                    'processed': True
                }

        self._save_sample_mapping(sample_mapping)

    def load_checkpoint(self) -> Tuple[Optional[set], Optional[int]]:
        """Load checkpoint if exists"""
        if self.checkpoint_file.exists():
            try:
                with open(self.checkpoint_file, 'rb') as f:
                    checkpoint_data = pickle.load(f)

                processed_indices = checkpoint_data.get('processed_indices', set())
                current_idx = checkpoint_data.get('current_idx', 0)
                is_smart_resume = checkpoint_data.get('smart_resume', False)

                if is_smart_resume:
                    print(f"Resuming smart checkpoint: {len(processed_indices)} samples already processed")
                else:
                    print(f"Found legacy checkpoint - will use smart matching instead")
                    # Don't use legacy checkpoint data as indices may not match
                    return set(), 0

                return processed_indices, current_idx
            except Exception as e:
                print(f"Failed to load checkpoint: {e}")
                return set(), 0

        return set(), 0

    def _initialize_results_df(self, df: pd.DataFrame) -> pd.DataFrame:
        """Initialize results dataframe with prediction columns"""
        results_df = df.copy()

        # Initialize prediction columns
        pred_columns = [
            'pred_direct_binary', 'pred_direct_type', 'pred_direct_degree', 'pred_direct_raw',
            'pred_cot_binary', 'pred_cot_type', 'pred_cot_degree', 'pred_cot_raw'
        ]

        for col in pred_columns:
            if col not in results_df.columns:
                results_df[col] = ''

        return results_df

    def is_sample_processed(self, df: pd.DataFrame, idx: int) -> bool:
        """Check if a sample has been fully processed"""
        required_cols = ['pred_direct_binary', 'pred_cot_binary']

        for col in required_cols:
            if col not in df.columns:
                return False
            value = df.loc[idx, col]
            if pd.isna(value) or str(value).strip() == '' or str(value).lower() == 'nan':
                return False

        return True

    def cleanup_checkpoint(self):
        """Remove checkpoint files after successful completion"""
        try:
            files_to_remove = [
                self.checkpoint_file,
                self.progress_file,
                self.sample_mapping_file
            ]

            for file_path in files_to_remove:
                if file_path.exists():
                    file_path.unlink()

            print(f"Cleaned up checkpoint files for {self.language} ({self.experiment_type})")
        except Exception as e:
            print(f"Failed to cleanup checkpoints: {e}")

    def get_smart_resume_stats(self, current_df: pd.DataFrame) -> Dict:
        """Get statistics about smart resume matching"""
        current_mapping = self._create_sample_mapping(current_df)
        existing_mapping = self._load_existing_sample_mapping()

        stats = {
            'current_dataset_size': len(current_df),
            'current_unique_samples': len(current_mapping),
            'existing_processed_samples': len(existing_mapping),
            'potentially_matchable': len(set(current_mapping.keys()) & set(existing_mapping.keys()))
        }

        return stats

# ========================== CLASSIFIER CLASS ==========================

class HallucinationClassifier:
    def __init__(self, model_name: str, experiment_type: str, use_flash_attn: bool = False):
        self.model, self.processor = setup_model(model_name, use_flash_attn)
        self.model_name = model_name.split('/')[-1]
        self.experiment_type = experiment_type

    def _prepare_conversation_audio(self, prompt: str, audio_path: str) -> List[Dict]:
        """Prepare conversation format for audio input"""
        return [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "audio", "audio": audio_path}
                ]
            }
        ]

    def _prepare_conversation_text(self, prompt: str, text_content: str) -> List[Dict]:
        """Prepare conversation format for text input"""
        return [
            {
                "role": "system",
                "content": [{"type": "text", "text": SYSTEM_PROMPT}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "text", "text": f"Text to analyze: {text_content}"}
                ]
            }
        ]

    def _generate_response(self, conversation: List[Dict]) -> str:
        """Generate model response with memory management"""
        try:
            # Clear GPU cache before processing
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Process conversation
            text = self.processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
            audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)

            # Prepare inputs
            inputs = self.processor(
                text=text,
                audio=audios,
                images=images,
                videos=videos,
                return_tensors="pt",
                padding=True,
                use_audio_in_video=False
            )
            inputs = inputs.to(self.model.device).to(self.model.dtype)

            # Generate with memory optimization
            with torch.no_grad():
                text_ids = self.model.generate(
                    **inputs,
                    return_audio=False,
                    max_new_tokens=256,
                    do_sample=False,
                    pad_token_id=self.processor.tokenizer.eos_token_id
                )

            # Clear inputs from memory
            del inputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Decode response
            response = self.processor.batch_decode(text_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

            # Clear text_ids from memory
            del text_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Extract assistant response
            if "<|im_start|>assistant" in response:
                response = response.split("<|im_start|>assistant")[-1].strip()

            return response

        except torch.cuda.OutOfMemoryError as e:
            print(f"CUDA Out of Memory: {e}")
            print("Clearing GPU cache and retrying...")

            # Aggressive memory cleanup
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()

            time.sleep(2)
            return ""

        except Exception as e:
            print(f"Error generating response: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return ""

    def classify_direct(self, input_data: str) -> Dict[str, str]:
        """Direct classification approach"""
        if self.experiment_type == 'audio':
            prompt = AUDIO_DIRECT_PROMPT
            conversation = self._prepare_conversation_audio(prompt, input_data)
        else:
            prompt = TEXT_DIRECT_PROMPT
            conversation = self._prepare_conversation_text(prompt, input_data)

        response = self._generate_response(conversation)
        return self._parse_response(response)

    def classify_cot(self, input_data: str) -> Dict[str, str]:
        """Chain-of-thought classification approach"""
        if self.experiment_type == 'audio':
            prompt = AUDIO_COT_PROMPT
            conversation = self._prepare_conversation_audio(prompt, input_data)
        else:
            prompt = TEXT_COT_PROMPT
            conversation = self._prepare_conversation_text(prompt, input_data)

        response = self._generate_response(conversation)
        return self._parse_response(response)

    def _parse_response(self, response: str) -> Dict[str, str]:
        """Parse model response to extract classification"""
        if not response:
            return {'binary': 'no', 'type': 'none', 'degree': 'none', 'raw_response': ''}

        try:
            # Look for JSON in response
            start = response.find('{')
            end = response.rfind('}') + 1
            if start != -1 and end != 0:
                json_str = response[start:end]
                result = json.loads(json_str)

                # Validate and normalize
                binary = result.get('binary', 'no').lower()
                type_val = result.get('type', 'none').lower()
                degree = result.get('degree', 'none').lower()

                # If no hallucination, set type and degree to none
                if binary == 'no':
                    type_val = 'none'
                    degree = 'none'

                return {
                    'binary': binary,
                    'type': type_val,
                    'degree': degree,
                    'raw_response': response
                }
        except:
            pass

        # Fallback parsing
        response_lower = response.lower()

        # Binary detection
        binary = 'yes' if any(phrase in response_lower for phrase in ['hallucination: yes', 'binary: yes', '"binary": "yes"']) else 'no'

        # Type detection
        type_val = 'none'
        if binary == 'yes':
            if 'factual_fabrication' in response_lower:
                type_val = 'factual_fabrication'
            elif 'factual_contradiction' in response_lower:
                type_val = 'factual_contradiction'
            elif 'contextual_inconsistency' in response_lower or 'contextual inconsistency' in response_lower:
                type_val = 'contextual_inconsistency'
            elif 'fabrication' in response_lower and 'factual_fabrication' not in response_lower:
                type_val = 'factual_fabrication'
            elif 'contradiction' in response_lower and 'factual_contradiction' not in response_lower:
                type_val = 'factual_contradiction'

        # Degree detection
        degree = 'none'
        if binary == 'yes':
            if 'severe' in response_lower:
                degree = 'severe'
            elif 'moderate' in response_lower:
                degree = 'moderate'
            elif 'mild' in response_lower:
                degree = 'mild'

        return {
            'binary': binary,
            'type': type_val,
            'degree': degree,
            'raw_response': response
        }

# ========================== EXPERIMENT RUNNER ==========================

def run_experiment_with_smart_resume(model_name, data_dict, experiment_type, force_restart=False, use_flash_attn=False):
    """Run classification experiment with smart checkpointing"""

    print(f"\nStarting {experiment_type} experiment with {model_name} (Smart Resume Enabled)")
    print("=" * 60)

    # Print GPU memory info before loading model
    print_gpu_memory_info()

    # Initialize classifier
    classifier = HallucinationClassifier(model_name, experiment_type, use_flash_attn)

    # Print GPU memory info after loading model
    print_gpu_memory_info()

    # Process each language
    for language, df in data_dict.items():
        print(f"\nProcessing {language} data ({len(df)} samples) - {experiment_type} mode")

        # Initialize smart checkpoint manager
        checkpoint_manager = SmartCheckpointManager(model_name, language, experiment_type)

        # Show smart resume statistics
        smart_stats = checkpoint_manager.get_smart_resume_stats(df)
        print(f"Dataset info: {smart_stats['current_dataset_size']} total samples, {smart_stats['current_unique_samples']} unique")
        print(f"Resume info: {smart_stats['existing_processed_samples']} previously processed, {smart_stats['potentially_matchable']} potentially matchable")

        # Load existing results with smart matching or initialize
        if force_restart:
            print("Force restart - ignoring existing results")
            results_df = checkpoint_manager._initialize_results_df(df)
            processed_indices = set()
        else:
            print("Using smart resume to match existing results...")
            results_df, processed_indices = checkpoint_manager.smart_load_existing_results(df)

        print(f"Smart resume result: {len(processed_indices)}/{len(df)} samples already processed")

        if len(processed_indices) == len(df):
            print(f"All samples for {language} ({experiment_type}) already processed!")
            continue

        # Create progress bar
        pbar = tqdm(total=len(df), initial=len(processed_indices),
                   desc=f"Processing {language} ({experiment_type})", unit="samples")

        # Process samples
        samples_since_checkpoint = 0
        checkpoint_interval = 5

        try:
            for idx in range(len(df)):
                # Skip if already processed
                if idx in processed_indices:
                    continue

                row = df.iloc[idx]

                try:
                    # Prepare input data based on experiment type
                    if experiment_type == 'audio':
                        audio_filename = row['filename']
                        language_folder = row['language_folder']
                        audio_path = os.path.join(BASE_DATA_DIR, language_folder, audio_filename)

                        if not os.path.exists(audio_path):
                            print(f"\nAudio file not found: {audio_path}")
                            pbar.update(1)
                            continue

                        input_data = audio_path
                    else:
                        if pd.isna(row['text']) or row['text'].strip() == '':
                            print(f"\nEmpty text content for sample {idx}")
                            pbar.update(1)
                            continue

                        input_data = row['text']

                    # Print memory info every 100 samples
                    if idx % 100 == 0 and torch.cuda.is_available():
                        memory_allocated = torch.cuda.memory_allocated() / 1024**3
                        memory_reserved = torch.cuda.memory_reserved() / 1024**3
                        print(f"\nGPU Memory: {memory_allocated:.1f}GB allocated, {memory_reserved:.1f}GB reserved")

                    # Direct classification
                    direct_result = classifier.classify_direct(input_data)
                    results_df.loc[idx, 'pred_direct_binary'] = direct_result['binary']
                    results_df.loc[idx, 'pred_direct_type'] = direct_result['type']
                    results_df.loc[idx, 'pred_direct_degree'] = direct_result['degree']
                    results_df.loc[idx, 'pred_direct_raw'] = direct_result['raw_response']

                    time.sleep(0.2)
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                    # CoT classification
                    cot_result = classifier.classify_cot(input_data)
                    results_df.loc[idx, 'pred_cot_binary'] = cot_result['binary']
                    results_df.loc[idx, 'pred_cot_type'] = cot_result['type']
                    results_df.loc[idx, 'pred_cot_degree'] = cot_result['degree']
                    results_df.loc[idx, 'pred_cot_raw'] = cot_result['raw_response']

                    # Mark as processed
                    processed_indices.add(idx)
                    samples_since_checkpoint += 1
                    pbar.update(1)

                    # Save checkpoint
                    if samples_since_checkpoint >= checkpoint_interval:
                        checkpoint_manager.save_checkpoint(results_df, processed_indices, idx)
                        samples_since_checkpoint = 0

                    # Memory cleanup
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

                    time.sleep(0.2)

                except Exception as e:
                    print(f"\nError processing sample {idx}: {e}")
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    pbar.update(1)
                    continue

        except KeyboardInterrupt:
            print(f"\nInterrupted! Saving progress...")
            checkpoint_manager.save_checkpoint(results_df, processed_indices, idx if 'idx' in locals() else 0)
            print(f"Progress saved. You can resume later.")
            return

        finally:
            pbar.close()

        # Final save
        checkpoint_manager.save_checkpoint(results_df, processed_indices, len(df)-1)

        # Generate and save summary
        summary = generate_summary_stats(results_df, language, classifier.model_name, experiment_type)
        summary_file = checkpoint_manager.output_dir / f"{language}_summary.json"
        with open(summary_file, 'w', encoding='utf-8') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)

        print(f"Completed {language} ({experiment_type}): {len(processed_indices)}/{len(df)} samples")
        print(f"Results saved to: {checkpoint_manager.results_file}")
        print(f"Summary saved to: {summary_file}")

        # Cleanup checkpoint files after successful completion
        if len(processed_indices) == len(df):
            checkpoint_manager.cleanup_checkpoint()

def generate_summary_stats(df: pd.DataFrame, language: str, model_name: str, experiment_type: str) -> Dict:
    """Generate summary statistics for the results"""
    summary = {
        'language': language,
        'model': model_name,
        'experiment_type': experiment_type,
        'total_samples': len(df),
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }

    for approach in ['direct', 'cot']:
        # Binary accuracy
        binary_col = f'pred_{approach}_binary'
        if binary_col in df.columns:
            binary_yes = (df[binary_col] == 'yes').sum()
            binary_accuracy = binary_yes / len(df) if len(df) > 0 else 0

            # Type distribution
            type_col = f'pred_{approach}_type'
            type_dist = df[type_col].value_counts().to_dict() if type_col in df.columns else {}

            # Degree distribution
            degree_col = f'pred_{approach}_degree'
            degree_dist = df[degree_col].value_counts().to_dict() if degree_col in df.columns else {}

            summary[approach] = {
                'binary_yes_predictions': int(binary_yes),
                'binary_accuracy': float(binary_accuracy),
                'type_distribution': type_dist,
                'degree_distribution': degree_dist
            }

    return summary

# ========================== VALIDATION AND UTILITIES ==========================

def validate_smart_resume(model_name: str, language: str, experiment_type: str):
    """Validate that smart resume will work correctly"""
    print(f"Validating smart resume for {model_name} - {language} - {experiment_type}")
    print("-" * 50)

    # Load both datasets
    data_dict_old = load_all_transcription_data(filter_hallucinated_only=True)
    data_dict_new = load_all_transcription_data(filter_hallucinated_only=False)

    if language not in data_dict_old or language not in data_dict_new:
        print(f"Language {language} not found in datasets")
        return 0, 0

    old_df = data_dict_old[language]
    new_df = data_dict_new[language]

    print(f"Old dataset (hallucinated only): {len(old_df)} samples")
    print(f"New dataset (all samples): {len(new_df)} samples")

    # Create checkpoint manager and check matching
    checkpoint_manager = SmartCheckpointManager(model_name, language, experiment_type)

    # Create sample mappings
    old_mapping = checkpoint_manager._create_sample_mapping(old_df)
    new_mapping = checkpoint_manager._create_sample_mapping(new_df)

    # Find matches
    matches = set(old_mapping.keys()) & set(new_mapping.keys())

    print(f"Unique samples in old dataset: {len(old_mapping)}")
    print(f"Unique samples in new dataset: {len(new_mapping)}")
    print(f"Matching samples: {len(matches)}")
    print(f"Match rate: {len(matches)/len(old_mapping)*100:.1f}%")

    if len(matches) < len(old_mapping) * 0.9:
        print("WARNING: Low match rate detected. Some samples may not transfer correctly.")

        unmatched = set(old_mapping.keys()) - matches
        print(f"\nExample unmatched samples (first 3):")
        for i, key in enumerate(list(unmatched)[:3]):
            old_idx = old_mapping[key]
            old_sample = old_df.iloc[old_idx]
            print(f"  Sample {i+1}: filename='{old_sample.get('filename', 'N/A')}', text_start='{str(old_sample.get('text', ''))[:50]}'")
    else:
        print("Good match rate - smart resume should work well!")

    return len(matches), len(old_mapping)

def show_progress_summary():
    """Show current progress for all experiments"""
    print("EXPERIMENT PROGRESS SUMMARY")
    print("=" * 60)

    checkpoint_base = Path(CHECKPOINT_DIR)
    if not checkpoint_base.exists():
        print("No experiments in progress.")
        return

    for model_dir in checkpoint_base.iterdir():
        if not model_dir.is_dir():
            continue

        print(f"\nModel: {model_dir.name}")
        print("-" * 50)

        for exp_type_dir in model_dir.iterdir():
            if not exp_type_dir.is_dir():
                continue

            print(f"  Experiment Type: {exp_type_dir.name}")
            print("  " + "-" * 40)

            for progress_file in exp_type_dir.glob("*_progress.json"):
                try:
                    with open(progress_file, 'r') as f:
                        progress = json.load(f)

                    lang = progress.get('language', 'unknown')
                    processed = progress.get('processed_count', 0)
                    total = progress.get('total_count', 0)
                    percentage = progress.get('progress_percentage', 0)
                    last_updated = progress.get('last_updated', 'unknown')

                    print(f"    {lang:10} | {processed:4d}/{total:4d} ({percentage:5.1f}%) | {last_updated}")

                except Exception as e:
                    print(f"    Error reading {progress_file}: {e}")

def print_gpu_memory_info():
    """Print current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        free = total - allocated
        print(f"GPU Memory: {allocated:.1f}GB/{total:.1f}GB used ({reserved:.1f}GB reserved, {free:.1f}GB free)")
    else:
        print("CUDA not available")

def merge_all_results():
    """Merge all results into single files per model and experiment type"""
    for model_dir in Path(OUTPUT_DIR).iterdir():
        if not model_dir.is_dir():
            continue

        print(f"Merging results for {model_dir.name}")

        for exp_type_dir in model_dir.iterdir():
            if not exp_type_dir.is_dir():
                continue

            print(f"  Experiment Type: {exp_type_dir.name}")

            all_results = []
            for result_file in exp_type_dir.glob("*_results.csv"):
                df = pd.read_csv(result_file, encoding='utf-8')
                all_results.append(df)

            if all_results:
                merged_df = pd.concat(all_results, ignore_index=True)
                merged_file = exp_type_dir / "all_languages_results.csv"
                merged_df.to_csv(merged_file, index=False, encoding='utf-8')
                print(f"    Saved merged results: {merged_file}")

# ========================== MAIN EXECUTION ==========================

def main_with_smart_resume(experiment_types: List[str] = None, force_restart: bool = False,
                          use_flash_attn: bool = False, filter_hallucinated_only: bool = False,
                          validate_first: bool = True):
    """Main execution function with smart resume capability"""

    print("Hallucination Classification Experiment with Smart Resume")
    print("=" * 60)

    if experiment_types is None:
        experiment_types = EXPERIMENT_TYPES

    # Validate experiment types
    valid_types = [exp_type for exp_type in experiment_types if exp_type in EXPERIMENT_TYPES]
    if not valid_types:
        print(f"Invalid experiment types. Valid options: {EXPERIMENT_TYPES}")
        return

    print(f"Experiment types to run: {valid_types}")

    # Show data filtering mode
    if filter_hallucinated_only:
        print("Data mode: Hallucinated samples only")
    else:
        print("Data mode: All samples (hallucinated + non-hallucinated)")

    # Load all transcription data
    print("\nLoading transcription data...")
    data_dict = load_all_transcription_data(filter_hallucinated_only=filter_hallucinated_only)

    if not data_dict:
        print("No data loaded. Please check file paths.")
        return

    # Print data summary
    total_samples = sum(len(df) for df in data_dict.values())
    print(f"\nTotal samples to process: {total_samples}")
    for lang, df in data_dict.items():
        if 'hallucination' in df.columns:
            hallucinated_count = (df['hallucination'].str.lower() == 'yes').sum()
            non_hallucinated_count = (df['hallucination'].str.lower() == 'no').sum()
            print(f"  - {lang}: {len(df)} samples (hallucinated: {hallucinated_count}, non-hallucinated: {non_hallucinated_count})")
        else:
            print(f"  - {lang}: {len(df)} samples")

    # Validate smart resume if not force restart and not hallucinated only
    if validate_first and not force_restart and not filter_hallucinated_only:
        print("\nValidating smart resume compatibility...")
        for model_name in MODELS:
            for exp_type in valid_types:
                for lang in data_dict.keys():
                    try:
                        matches, total = validate_smart_resume(model_name, lang, exp_type)
                        if matches < total * 0.9:
                            response = input(f"\nLow match rate for {model_name}-{lang}-{exp_type}. Continue anyway? (y/n): ")
                            if response.lower() != 'y':
                                print("Aborting...")
                                return
                    except Exception as e:
                        print(f"Validation failed for {model_name}-{lang}-{exp_type}: {e}")

    # Create output directories
    Path(OUTPUT_DIR).mkdir(exist_ok=True)
    Path(CHECKPOINT_DIR).mkdir(exist_ok=True)

    # Run experiments for each model and experiment type
    for model_name in MODELS:
        for experiment_type in valid_types:
            try:
                run_experiment_with_smart_resume(model_name, data_dict, experiment_type, force_restart, use_flash_attn)
            except Exception as e:
                print(f"Failed to run {experiment_type} experiment with {model_name}: {e}")
                continue

    print(f"\nAll experiments completed! Results saved in: {OUTPUT_DIR}")

# ========================== EXECUTION ==========================

if __name__ == "__main__":
    # Run with smart resume for all data (automatically matches existing hallucinated results)
    main_with_smart_resume(
        experiment_types=['audio'],
        force_restart=False,
        use_flash_attn=False,
        filter_hallucinated_only=False,  # Process ALL samples
        validate_first=True  # Validate compatibility first
    )

    # Other options:
    # main_with_smart_resume(experiment_types=['audio', 'text'], force_restart=False, use_flash_attn=False, filter_hallucinated_only=False, validate_first=True)
    # main_with_smart_resume(experiment_types=['text'], force_restart=True, use_flash_attn=False, filter_hallucinated_only=False, validate_first=False)
    # show_progress_summary()
    # merge_all_results()

Hallucination Classification Experiment with Smart Resume
Experiment types to run: ['audio']
Data mode: All samples (hallucinated + non-hallucinated)

Loading transcription data...

Loading english transcriptions from English/ (all samples)...
Loaded /content/drive/MyDrive/ICASSP_Hallucinaton/Audio_data/English/transcriptions_english_full.csv
Original data shape: (3965, 5)
Columns: ['filename', 'text', 'hallucination', 'hallucination_type', 'hallucination_level']
Processing all data: (3965, 5)

Loading kazakh transcriptions from Kazakh/ (all samples)...
Loaded /content/drive/MyDrive/ICASSP_Hallucinaton/Audio_data/Kazakh/transcriptions_kazakh_full_with_corrections.csv
Original data shape: (3977, 5)
Columns: ['filename', 'text', 'hallucination', 'hallucination_type', 'hallucination_level']
Processing all data: (3977, 5)

Loading russian transcriptions from Russian/ (all samples)...
Loaded /content/drive/MyDrive/ICASSP_Hallucinaton/Audio_data/Russian/transcriptions_russian_full.csv
Origin

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully
GPU Memory: 8.8GB/39.6GB used (22.4GB reserved, 30.8GB free)

Processing english data (3965 samples) - audio mode
Dataset info: 3965 total samples, 3965 unique
Resume info: 0 previously processed, 0 potentially matchable
Using smart resume to match existing results...
Current dataset has 3965 unique samples
Found existing results with 2627 samples
Successfully matched 2627 processed samples
Found 0 samples that couldn't be matched or weren't processed
Smart resume result: 2627/3965 samples already processed


Processing english (audio):  66%|██████▋   | 2627/3965 [00:00<?, ?samples/s]

...


GPU Memory: 8.8GB allocated, 11.2GB reserved


Processing russian (audio):  99%|█████████▊| 4009/4067 [2:31:17<07:04,  7.31s/samples]


GPU Memory: 8.8GB allocated, 11.2GB reserved


Processing russian (audio): 100%|██████████| 4067/4067 [2:39:04<00:00,  7.59s/samples]


Completed russian (audio): 4067/4067 samples
Results saved to: hallucination_results/Qwen2.5-Omni-3B/audio/russian_results.csv
Summary saved to: hallucination_results/Qwen2.5-Omni-3B/audio/russian_summary.json
Cleaned up checkpoint files for russian (audio)

All experiments completed! Results saved in: hallucination_results
